In [36]:
# Install zstd dependency
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background and wait for it to be ready
import subprocess
import time

# Start ollama serve as a background process
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Give the server some time to start
print("Waiting for Ollama server to start...")
time.sleep(10) # Wait for 10 seconds. Adjust if necessary.
print("Ollama server should be running now.")

# Pull and run Qwen2.5-7B
!ollama run qwen2.5:7b "Hello! Introduce yourself."

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Waiting for Ollama server to start...
Ollama server should be running now.
Hello! I'm Qwen, an AI assistant created by Alibaba Cloud. I'm here to help
help with a wide range of tasks, from answering questions and providing inf
information on various topics, to assisting with writing, translating langu
languages, and more. My goal

In [37]:
# Cell 2
!ollama pull qwen2.5vl:7b

In [38]:
# Cell 1
!pip install -q ollama pdf2image pillow
!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (24.02.0-1ubuntu9.9).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [39]:
# Cell 3
from google.colab import files
import os

print("Please upload a sample House BL document (PDF or Image):")
uploaded = files.upload()

# Get the filename of the uploaded file
file_name = list(uploaded.keys())[0]
print(f"File '{file_name}' successfully uploaded and ready for extraction.")

Please upload a sample House BL document (PDF or Image):


Saving Sample_HBL.pdf to Sample_HBL (2).pdf
File 'Sample_HBL (2).pdf' successfully uploaded and ready for extraction.


In [44]:
!ollama pull qwen2.5vl:3b

In [45]:
# Cell 4 (Memory-Optimized with 3B Model)
import json
import os
import ollama
from pdf2image import convert_from_path
from PIL import Image

def extract_hbl_number(file_path):
    print(f"Processing '{file_path}'...")

    image_path = None
    temporary_image_created = False

    # 1. Convert PDF to a modest resolution to prevent memory spikes
    if file_path.lower().endswith('.pdf'):
        print(f"Converting PDF to lightweight image...")
        try:
            images = convert_from_path(file_path, first_page=1, last_page=1, dpi=100)

            # Keep image dimensions moderate
            img = images[0]
            img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)

            image_path = "temp_hbl.jpg"
            img.save(image_path, "JPEG", quality=70)
            temporary_image_created = True
        except Exception as e:
            print(f"PDF conversion error: {e}")
            image_path = file_path
    else:
        try:
            img = Image.open(file_path)
            img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
            image_path = "temp_hbl.jpg"
            img.convert('RGB').save(image_path, "JPEG", quality=70)
            temporary_image_created = True
        except Exception as e:
            image_path = file_path

    # 2. Concise Prompt
    prompt = """
    Extract the House Bill of Lading Number (HBL No. / B/L No.) from this document.
    Return ONLY JSON:
    {"hbl_number": "string or null", "confidence": "high/medium/low"}
    """

    # 3. Call 3B Model with conservative parameters
    try:
        print("Running 3B vision model inference...")
        response = ollama.chat(
            model='qwen2.5vl:3b',  # Switched to 3B model to avoid GPU OOM crashes
            messages=[{
                'role': 'user',
                'content': prompt,
                'images': [image_path]
            }],
            options={
                'num_ctx': 4096,
                'num_predict': 64,
                'temperature': 0.0
            }
        )
        result_text = response['message']['content']
    except Exception as e:
        print(f"Ollama API Error: {e}")
        result_text = json.dumps({"hbl_number": None, "confidence": "low", "error": str(e)})

    # Cleanup
    if temporary_image_created and os.path.exists("temp_hbl.jpg"):
        os.remove("temp_hbl.jpg")

    return result_text

In [46]:
# Cell 5
# file_name is automatically passed from Cell 3
result = extract_hbl_number(file_name)

print("\n--- EXTRACTION RESULT ---")
print(result)

Processing 'Sample_HBL (2).pdf'...
Converting PDF to lightweight image...
Running 3B vision model inference...

--- EXTRACTION RESULT ---
```json
{
  "hbl_number": "HPS1201K08-221",
  "confidence": "high"
}
```
